In [60]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as dataset

import numpy as np
import matplotlib.pyplot as plt

In [61]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, causal=False) -> None:
    super().__init__()

    self.d_k = d_k
    self.n_heads = n_heads

    self.key = nn.Linear(d_model, d_k * n_heads)
    self.query = nn.Linear(d_model, d_k * n_heads)
    self.value = nn.Linear(d_model, d_k * n_heads)

    self.fc = nn.Linear(d_k * n_heads, d_model)

    self.causal = causal
    if causal:
      cm = torch.tril(torch.ones(max_len, max_len))
      self.register_buffer(
        'causal_mask',
        cm.view(1, 1, max_len, max_len)
    )


  def forward(self, q, k, v, pad_mask=None):
    q = self.query(q)   # (N, T, d_model) -> (N, T, h*d_k)
    k = self.key(k)   # (N, T, d_model) -> (N, T, h*d_k)
    v = self.value(v)   # (N, T, d_model) -> (N, T, h*d_k)

    N = q.shape[0]
    T_output = q.shape[1]
    T_input = k.shape[1]

    # change the shape
    # (N, T, h, d_k) -> (N, h, T, d_k)
    # in order for matrix multiply to work properly
    q = q.view(N, T_output, self.n_heads, self.d_k).transpose(1, 2)
    k = k.view(N, T_input, self.n_heads, self.d_k).transpose(1, 2)
    v = v.view(N, T_input, self.n_heads, self.d_k).transpose(1, 2)

    # Compute attention weights
    # (N, h, T, d_k) x (N, h, d_k, T) -> (N, h, T, T)
    attn_scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)
    if pad_mask is not None:
      attn_scores = attn_scores.masked_fill(pad_mask[:, None, None, :] == 0, float('-inf'))

    if self.causal:
      attn_scores = attn_scores.masked_fill(self.causal_mask[:, :, :T_output, :T_input] == 0, float('-inf'))

    attn_weights = F.softmax(attn_scores, dim=-1)

    # Compute attention weights values
    # (N, h, T, T) x (N, h, T, d_k) -> (N, h, T, d_k)
    A = attn_weights @ v

    # Reshape it back before final linear layer
    A = A.transpose(1, 2) # (N, T, h, d_k)
    A = A.contiguous().view(N, T_output, self.d_k * self.n_heads) # (N, T, h*d_k)

    # projection
    return self.fc(A)

In [62]:
class EncoderBlock(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, dropout_prob=0.1) -> None:
    super().__init__()

    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)
    self.mha = MultiHeadAttention(d_k, d_model, n_heads, max_len, causal=False)
    self.ann = nn.Sequential(
        nn.Linear(d_model, d_model * 4),
        nn.GELU(),
        nn.Linear(d_model * 4, d_model),
        nn.Dropout(dropout_prob)
    )
    self.dropout = nn.Dropout(p=dropout_prob)

  def forward(self, x, mask=None):
    x = self.ln1(x + self.mha(x, x, x, mask))
    x = self.ln2(x + self.ann(x))
    x = self.dropout(x)
    return x

In [63]:
class DecoderBlock(nn.Module):
  def __init__(self, d_k, d_model, n_heads, max_len, dropout_prob=0.1) -> None:
    super().__init__()

    self.ln1 = nn.LayerNorm(d_model)
    self.ln2 = nn.LayerNorm(d_model)
    self.ln3 = nn.LayerNorm(d_model)
    self.mha1 = MultiHeadAttention(d_k, d_model, n_heads, max_len, causal=True)
    self.mha2 = MultiHeadAttention(d_k, d_model, n_heads, max_len, causal=False)
    self.ann = nn.Sequential(
        nn.Linear(d_model, d_model * 4),
        nn.GELU(),
        nn.Linear(d_model * 4, d_model),
        nn.Dropout(dropout_prob)
    )
    self.dropout = nn.Dropout(p=dropout_prob)

  def forward(self, enc_output, dec_input, enc_mask=None, dec_mask=None):
    # self-attention on decoder input
    x = self.ln1(dec_input + self.mha1(dec_input, dec_input, dec_input, dec_mask))

    # multi-head attention including incoder output
    x = self.ln2(x + self.mha2(x, enc_output, enc_output, enc_mask))

    x = self.ln3(x + self.ann(x))
    x = self.dropout(x)
    return x

In [64]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=2048, dropout_prob=0.1) -> None:
    super().__init__()

    self.dropout = nn.Dropout(p=dropout_prob)

    position = torch.arange(max_len).unsqueeze(1)
    exp_term = torch.arange(0, d_model, 2)
    div_term = torch.exp(exp_term * (-math.log(10000.0)/d_model))
    pe = torch.zeros(1, max_len, d_model)
    pe[0, :, 0::2] = torch.sin(position * div_term)
    pe[0, :, 1::2] = torch.cos(position * div_term)
    self.register_buffer('pe', pe)

  def forward(self, x):
    # x.shape (N, T, D)
    x = x + self.pe[:, :x.size(1), :]
    return self.dropout(x)

In [65]:
class Encoder(nn.Module):
  def __init__(self, vocab_size, max_len, d_k, d_model, n_heads, n_layers,
               #n_classes,
               dropout_prob) -> None:
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, d_model)
    self.pos_encoding = PositionalEncoding(d_model, max_len, dropout_prob)
    transformer_blocks = [
        EncoderBlock(d_k, d_model, n_heads, max_len, dropout_prob) for _ in range(n_layers)]
    self.transformer_blocks = nn.Sequential(*transformer_blocks)
    self.ln = nn.LayerNorm(d_model)
    #self.fc = nn.Linear(d_model, n_classes)

  def forward(self, x, mask=None):
    x = self.embedding(x)
    x = self.pos_encoding(x)

    for block in self.transformer_blocks:
      x = block(x, mask)

    # many-to-one x has shape (N x T x D) -> (N x D)
    #x = x[:, 0, :]

    x = self.ln(x)
    #x = self.fc(x)
    return x

In [66]:
class Decoder(nn.Module):
  def __init__(self, vocab_size, max_len, d_k, d_model, n_heads, n_layers, dropout_prob) -> None:
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, d_model)
    self.pos_encoding = PositionalEncoding(d_model, max_len, dropout_prob)
    decoder_blocks = [
        DecoderBlock(d_k, d_model, n_heads, max_len, dropout_prob) for _ in range(n_layers)]
    self.decoder_blocks = nn.Sequential(*decoder_blocks)
    self.ln = nn.LayerNorm(d_model)
    self.fc = nn.Linear(d_model, vocab_size)

  def forward(self, enc_output, dec_input, enc_mask=None, dec_mask=None):

    # print("=" * 50)
    # print("DEBUGGING DECODER INPUT:")
    # print(f"dec_input shape: {dec_input.shape}")
    # print(f"dec_input dtype: {dec_input.dtype}")
    # print(f"dec_input min value: {dec_input.min().item()}")
    # print(f"dec_input max value: {dec_input.max().item()}")
    # print(f"Embedding vocab size: {self.embedding.num_embeddings}")
    # print(f"Valid token range: 0 to {self.embedding.num_embeddings - 1}")

    x = self.embedding(dec_input)
    x = self.pos_encoding(x)

    for block in self.decoder_blocks:
      x = block(enc_output, x, enc_mask, dec_mask)

    # many-to-one x has shape (N x T x D) -> (N x D)
    # x = x[:, 0, :] - this is for encoder

    x = self.ln(x)
    x = self.fc(x) # many-to-many
    return x

In [67]:
class Transformer(nn.Module):
  def __init__(self, encoder, decoder) -> None:
    super().__init__()

    self.encoder = encoder
    self.decoder = decoder

  def forward(self, enc_input, dec_input, enc_mask, dec_mask):
    enc_output = self.encoder(enc_input, enc_mask)
    dec_output = self.decoder(enc_output, dec_input, enc_mask, dec_mask)

    return dec_output

In [68]:
encoder = Encoder(vocab_size=20_000,
                  max_len=512,
                  d_k=16,
                  d_model=64,
                  n_heads=4,
                  n_layers=2,
                  dropout_prob=0.1)

decoder = Decoder(vocab_size=10_000,
                  max_len=512,
                  d_k=16,
                  d_model=64,
                  n_heads=4,
                  n_layers=2,
                  dropout_prob=0.1)

transformer = Transformer(encoder, decoder)

In [69]:
# print("Before .to():", encoder.shape, encoder.dtype, encoder.device)
# print("Contains NaN:", torch.isnan(encoder).any())

# import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [70]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
encoder.to(device)
decoder.to(device)

cuda:0


Decoder(
  (embedding): Embedding(10000, 64)
  (pos_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (decoder_blocks): Sequential(
    (0): DecoderBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln3): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mha1): MultiHeadAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
      )
      (mha2): MultiHeadAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
     

In [71]:
xe = np.random.randint(0, 20_000, size=(8, 512))
xe_t = torch.tensor(xe).to(device)

In [72]:
xd = np.random.randint(0, 10_000, size=(8, 256))
xd_t = torch.tensor(xd).to(device)

In [73]:
maske = np.ones((8, 512))
maske[:, 256:] = 0
maske_t = torch.tensor(maske).to(device)

maskd = np.ones((8, 256))
maskd[:, 128:] = 0
maskd_t = torch.tensor(maskd).to(device)

In [74]:
out = transformer(xe_t, xd_t, maske_t, maskd_t)
out.shape

torch.Size([8, 256, 10000])

In [75]:
out

tensor([[[ 0.3081, -0.5561,  0.1978,  ..., -0.0113,  0.5756, -0.4969],
         [ 0.7100, -0.2309,  0.6898,  ...,  0.1522, -0.7185, -0.2864],
         [ 0.1703, -0.6504,  1.5483,  ..., -0.0789, -0.5830,  0.7344],
         ...,
         [-0.5091, -0.1354,  0.7074,  ...,  0.4613, -0.1116,  0.4164],
         [-0.7181, -0.3899,  0.4032,  ..., -0.5169, -0.6123,  0.4438],
         [-0.6104, -0.1007,  0.6239,  ..., -0.4077, -0.3624,  0.2935]],

        [[ 1.0067, -0.3323, -0.0906,  ...,  0.0634, -0.5587,  1.1272],
         [-0.7608, -0.2550, -0.3128,  ...,  0.1430,  0.9219, -0.2097],
         [-0.2828, -0.5342, -0.2304,  ...,  0.0036, -0.7773, -0.4977],
         ...,
         [ 0.4145, -0.5845,  0.3189,  ...,  0.7242, -0.6697,  0.4110],
         [-0.7247,  0.1283, -0.3119,  ...,  0.3369, -0.9422,  0.8594],
         [-0.2492, -0.3139, -0.2478,  ..., -0.3521,  0.4127,  0.5635]],

        [[ 0.1621, -0.5225, -0.1245,  ...,  1.7298, -0.0873,  0.6659],
         [ 0.6134, -0.3612,  0.2691,  ..., -0

In [76]:
!head rus.txt

Go.	Марш!	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #1159202 (shanghainese)
Go.	Иди.	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #5898247 (marafon)
Go.	Идите.	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #5898250 (marafon)
Hi.	Здравствуйте.	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #402127 (odexed)
Hi.	Привет!	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #466968 (katjka)
Hi.	Хай.	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #467233 (timsa)
Hi.	Здрасте.	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #3803577 (marafon)
Hi.	Здоро́во!	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #3854188 (marafon)
Hi.	Приветик!	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #7234283 (marafon)
Run!	Беги!	CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #1569978 (Biga)


In [77]:
import pandas as pd

df = pd.read_csv('rus.txt', sep="\t", header=None)
df.head()

,0,1,2
0,Go.,Марш!,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
1,Go.,Иди.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
2,Go.,Идите.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
3,Hi.,Здравствуйте.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
4,Hi.,Привет!,CC-BY 2.0 (France) Attribution: tatoeba.org #5...


In [78]:
df.shape

(34723, 3)

In [79]:
df = df.iloc[:30_000]

In [80]:
df.columns = ['en', 'ru', 'd']

In [81]:
df

,en,ru,d
0,Go.,Марш!,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
1,Go.,Иди.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
2,Go.,Идите.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
3,Hi.,Здравствуйте.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
4,Hi.,Привет!,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
...,...,...,...
29995,Tom is European.,Том - европеец.,CC-BY 2.0 (France) Attribution: tatoeba.org #1...
29996,Tom is a Cancer.,Том - Рак.,CC-BY 2.0 (France) Attribution: tatoeba.org #7...
29997,Tom is a Gemini.,Том – Близнецы.,CC-BY 2.0 (France) Attribution: tatoeba.org #7...
29998,Tom is a Muslim.,Том - мусульманин.,CC-BY 2.0 (France) Attribution: tatoeba.org #1...


In [82]:
df = df.drop('d', axis=1)

In [83]:
df

,en,ru
0,Go.,Марш!
1,Go.,Иди.
2,Go.,Идите.
3,Hi.,Здравствуйте.
4,Hi.,Привет!
...,...,...
29995,Tom is European.,Том - европеец.
29996,Tom is a Cancer.,Том - Рак.
29997,Tom is a Gemini.,Том – Близнецы.
29998,Tom is a Muslim.,Том - мусульманин.


In [84]:
df.to_csv('rus.csv', index=None)

In [85]:
!head rus.csv

en,ru
Go.,Марш!
Go.,Иди.
Go.,Идите.
Hi.,Здравствуйте.
Hi.,Привет!
Hi.,Хай.
Hi.,Здрасте.
Hi.,Здоро́во!
Hi.,Приветик!


In [86]:
!pip install transformers datasets sentencepiece sacremoses

In [87]:
from datasets import load_dataset
raw_dataset = load_dataset('csv', data_files='rus.csv')

Generating train split: 0 examples [00:00, ? examples/s]

In [88]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'ru'],
        num_rows: 30000
    })
})

In [89]:
split = raw_dataset['train'].train_test_split(test_size=0.3, seed=42)
split

DatasetDict({
    train: Dataset({
        features: ['en', 'ru'],
        num_rows: 21000
    })
    test: Dataset({
        features: ['en', 'ru'],
        num_rows: 9000
    })
})

In [90]:
from transformers import AutoTokenizer

model_checkpoint = 'Helsinki-NLP/opus-mt-en-ru'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [91]:
en_sentence = split['train'][0]['en']
ru_sentence = split['train'][0]['ru']

inputs = tokenizer(en_sentence)
targets = tokenizer(text_target=ru_sentence)

tokenizer.convert_ids_to_tokens(targets['input_ids'])

['▁Не', '▁', 'двига', 'й', 'ся', '!', '</s>']

In [92]:
en_sentence, ru_sentence

("Don't move!", 'Не двигайся!')

In [93]:
max_input_length = 128
max_target_length = 128

In [94]:
def preprocess_function(batch):
  model_inputs = tokenizer(batch['en'], max_length=max_input_length, truncation=True)
  labels = tokenizer(text_target=batch['ru'], max_length=max_target_length, truncation=True)

  model_inputs['labels'] = labels['input_ids']
  return model_inputs

In [95]:
tokenized_datasets = split.map(preprocess_function, batched=True, remove_columns=split['train'].column_names)

Map:   0%|          | 0/21000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

In [96]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 9000
    })
})

In [97]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer)

In [98]:
batch = data_collator([tokenized_datasets['train'][i] for i in range(0, 5)])
batch.keys()

KeysView({'input_ids': tensor([[ 1519,    18,    45,  2728,    56,     0, 62517],
        [11875,    55, 50869,     3,     0, 62517, 62517],
        [  961,    33,   182,   108,    19,     0, 62517],
        [  103,    18,   153,   195,  2007,     3,     0],
        [  160,  4894, 13617,     3,     0, 62517, 62517]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0, 0]]), 'labels': tensor([[  327,    21,  9015,    48,    93,    56,     0],
        [31836,  1723,  4029,   387,     3,     0,  -100],
        [  784,   247,  2945,   224,    19,     0,  -100],
        [  206,  3211,  2920,   106,     3,     0,  -100],
        [  261,   907, 13239,   357,  4047,     3,     0]])})

In [99]:
batch['input_ids']

tensor([[ 1519,    18,    45,  2728,    56,     0, 62517],
        [11875,    55, 50869,     3,     0, 62517, 62517],
        [  961,    33,   182,   108,    19,     0, 62517],
        [  103,    18,   153,   195,  2007,     3,     0],
        [  160,  4894, 13617,     3,     0, 62517, 62517]])

In [100]:
batch['attention_mask']

tensor([[1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0, 0]])

In [101]:
batch['labels']

tensor([[  327,    21,  9015,    48,    93,    56,     0],
        [31836,  1723,  4029,   387,     3,     0,  -100],
        [  784,   247,  2945,   224,    19,     0,  -100],
        [  206,  3211,  2920,   106,     3,     0,  -100],
        [  261,   907, 13239,   357,  4047,     3,     0]])

In [102]:
tokenizer.all_special_ids

[0, 1, 62517]

In [103]:
tokenizer.all_special_tokens

['</s>', '<unk>', '<pad>']

In [104]:
tokenizer('<pad>')

{'input_ids': [62517, 0], 'attention_mask': [1, 1]}

In [105]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    tokenized_datasets['train'],
    shuffle=True,
    batch_size=32,
    collate_fn=data_collator)

valid_loader = DataLoader(
    tokenized_datasets['test'],
    shuffle=True,
    batch_size=32,
    collate_fn=data_collator)

In [106]:
for batch in train_loader:
  for k, v in batch.items():
    print("k: ", k, "v.shape: ", v.shape)
  break

k:  input_ids v.shape:  torch.Size([32, 9])
k:  attention_mask v.shape:  torch.Size([32, 9])
k:  labels v.shape:  torch.Size([32, 9])


In [107]:
tokenizer.vocab_size

62518

In [108]:
tokenizer.decode(62517)

'<pad>'

In [109]:
tokenizer.add_special_tokens({"cls_token": "<s>"})

1

In [110]:
tokenizer('<s>')

{'input_ids': [62518, 0], 'attention_mask': [1, 1]}

In [111]:
tokenizer.decode(tokenizer('<s>')['input_ids'])

'<s> </s>'

In [112]:
tokenizer.vocab_size

62518

In [113]:
encoder = Encoder(vocab_size=tokenizer.vocab_size + 1,
                  max_len=512,
                  d_k=16,
                  d_model=64,
                  n_heads=4,
                  n_layers=2,
                  dropout_prob=0.1)

decoder = Decoder(vocab_size=tokenizer.vocab_size + 1,
                  max_len=512,
                  d_k=16,
                  d_model=64,
                  n_heads=4,
                  n_layers=2,
                  dropout_prob=0.1)

transformer = Transformer(encoder, decoder)

In [114]:
# torch.cuda.empty_cache()

In [115]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
encoder.to(device)
decoder.to(device)

cuda:0


Decoder(
  (embedding): Embedding(62519, 64)
  (pos_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (decoder_blocks): Sequential(
    (0): DecoderBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ln3): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mha1): MultiHeadAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
      )
      (mha2): MultiHeadAttention(
        (key): Linear(in_features=64, out_features=64, bias=True)
        (query): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
     

In [116]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(transformer.parameters())

In [117]:
from datetime import datetime

def train(model, criterion, optimizer, train_loader, valid_loader, epochs):
  train_losses = np.zeros(epochs)
  test_losses = np.zeros(epochs)

  for it in range(epochs):
    model.train()
    t0 = datetime.now()
    train_loss = []
    for batch in train_loader:
      # move data to GPU
      batch = {k: v.to(device) for k, v in batch.items()}

      # zero the parameter gradients
      optimizer.zero_grad()

      enc_input = batch['input_ids']
      enc_mask = batch['attention_mask']
      targets = batch['labels']

      # shift targets forwards to get decoder_input
      dec_input = targets.clone().detach()
      dec_input = torch.roll(dec_input, shifts=1, dims=1)
      dec_input[:, 0] = 62_518 #65_001

      # also convert all -100 to pad token id
      dec_input = dec_input.masked_fill(dec_input == -100, tokenizer.pad_token_id)

      # make decoder input mask
      dec_mask = torch.ones_like(dec_input)
      dec_mask = dec_mask.masked_fill(dec_input == tokenizer.pad_token_id, 0)


      # forward pass
      outputs = model(enc_input, dec_input, enc_mask, dec_mask)
      loss = criterion(outputs.transpose(2, 1), targets)

      # backward and optimizer
      loss.backward()
      optimizer.step()
      train_loss.append(loss.item())

    # get average train loss
    train_loss = np.mean(train_loss)

    model.eval()
    test_loss = []


    for batch in valid_loader:
      # move data to GPU
      batch = {k: v.to(device) for k, v in batch.items()}

      # zero the parameter gradients
      # optimizer.zero_grad()

      enc_input = batch['input_ids']
      enc_mask = batch['attention_mask']
      targets = batch['labels']

      # shift targets forwards to get decoder_input
      dec_input = targets.clone().detach()
      dec_input = torch.roll(dec_input, shifts=1, dims=1)
      dec_input[:, 0] = 62_518 #65_001

      # also convert all -100 to pad token id
      dec_input = dec_input.masked_fill(dec_input == -100, tokenizer.pad_token_id)

      # make decoder input mask
      dec_mask = torch.ones_like(dec_input)
      dec_mask = dec_mask.masked_fill(dec_input == tokenizer.pad_token_id, 0)


      # forward pass
      outputs = model(enc_input, dec_input, enc_mask, dec_mask)
      loss = criterion(outputs.transpose(2, 1), targets)

      # backward and optimizer
      # loss.backward()
      # optimizer.step()
      test_loss.append(loss.item())

    # get average train loss
    test_loss = np.mean(test_loss)



    train_losses[it] = train_loss
    test_losses[it] = test_loss

    dt = datetime.now() - t0
    print(f'Epoch {it+1}/{epochs}, Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, Duration: {dt}')

  return train_losses, test_losses

In [118]:
train_losses, test_losses = train(transformer, criterion, optimizer, train_loader, valid_loader, epochs=15)

Epoch 1/15, Train loss: 4.5997, Test loss: 3.6360, Duration: 0:00:41.421356
Epoch 2/15, Train loss: 3.3271, Test loss: 3.0383, Duration: 0:00:17.446899
Epoch 3/15, Train loss: 2.8401, Test loss: 2.7154, Duration: 0:00:16.730103
Epoch 4/15, Train loss: 2.4950, Test loss: 2.4436, Duration: 0:00:16.873908
Epoch 5/15, Train loss: 2.2377, Test loss: 2.2606, Duration: 0:00:17.155071
Epoch 6/15, Train loss: 2.0323, Test loss: 2.1508, Duration: 0:00:23.051316
Epoch 7/15, Train loss: 1.8694, Test loss: 2.0516, Duration: 0:00:25.447713
Epoch 8/15, Train loss: 1.7463, Test loss: 1.9781, Duration: 0:00:16.822835
Epoch 9/15, Train loss: 1.6391, Test loss: 1.9409, Duration: 0:00:16.998151
Epoch 10/15, Train loss: 1.5485, Test loss: 1.8986, Duration: 0:00:16.981484
Epoch 11/15, Train loss: 1.4762, Test loss: 1.8641, Duration: 0:00:16.646314
Epoch 12/15, Train loss: 1.4126, Test loss: 1.8467, Duration: 0:00:17.261302
Epoch 13/15, Train loss: 1.3556, Test loss: 1.8334, Duration: 0:00:18.832940
Epoch 14

In [119]:
input_sentence = split['test'][10]['en']
input_sentence

'Tom stood, too.'

In [120]:
enc_input = tokenizer(input_sentence, return_tensors='pt')
enc_input

{'input_ids': tensor([[ 2226, 13269,     2,   846,     3,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

In [121]:
dec_input_str = '<s>'

dec_input = tokenizer(text_target=dec_input_str, return_tensors='pt')
dec_input

{'input_ids': tensor([[62518,     0]]), 'attention_mask': tensor([[1, 1]])}

In [122]:
enc_input.to(device)
dec_input.to(device)
output = transformer(enc_input['input_ids'],
                     dec_input['input_ids'][:, :-1],
                     enc_input['attention_mask'],
                     dec_input['attention_mask'][:, :-1])

output

tensor([[[ -1.1712, -15.9082,   8.6576,  ..., -14.9467, -15.8609, -14.0616]]],
       device='cuda:0', grad_fn=<ViewBackward0>)

In [123]:
output.shape # N x T x V

torch.Size([1, 1, 62519])

In [124]:
enc_output = encoder(enc_input['input_ids'], enc_input['attention_mask'])
enc_output.shape

torch.Size([1, 6, 64])

In [125]:
dec_output = decoder(enc_output,
                     dec_input['input_ids'][:, :-1],
                     enc_input['attention_mask'],
                     dec_input['attention_mask'][:, :-1])

dec_output.shape

torch.Size([1, 1, 62519])

In [126]:
torch.allclose(output, dec_output)

True

In [127]:
dec_input_ids = dec_input['input_ids'][:, :-1]
dec_attn_mask = dec_input['attention_mask'][:, :-1]

for _ in range(32):
  dec_output = decoder(enc_output,
                      dec_input_ids,
                      enc_input['attention_mask'],
                      dec_attn_mask)

  # choose the best value (or sample)
  prediction_id = torch.argmax(dec_output[:, -1, :], axis=-1)

  # append to decoder input
  dec_input_ids = torch.hstack((dec_input_ids, prediction_id.view(1, 1)))

  # recreate mask
  dec_attn_mask = torch.ones_like(dec_input_ids)

  # exit when reach </s>
  if prediction_id == 0:
    break


In [128]:
tokenizer.decode(dec_input_ids[0])

'<s>  Том тоже стоял.</s>'

In [129]:
split['test'][10]['ru']

'Том тоже стоял.'

In [145]:
def greedy_translate(input_sentence):
  # get encoder output first
  enc_input = tokenizer(input_sentence, return_tensors='pt').to(device)
  enc_output = encoder(enc_input['input_ids'], enc_input['attention_mask'])

  # setup initial decoder input
  #dec_input_ids = torch.tensor([[65_001]], device=device)
  dec_input_ids = torch.tensor([[62_518]], device=device)
  dec_attn_mask = torch.ones_like(dec_input_ids, device=device)

  # now do the decoder loop
  for _ in range(32):
    dec_output = decoder(enc_output,
                        dec_input_ids,
                        enc_input['attention_mask'],
                        dec_attn_mask)

    # choose the best value (or sample)
    prediction_id = torch.argmax(dec_output[:, -1, :], axis=-1)

    # append to decoder input
    dec_input_ids = torch.hstack((dec_input_ids, prediction_id.view(1, 1)))

    # recreate mask
    dec_attn_mask = torch.ones_like(dec_input_ids)

    # exit when reach </s>
    if prediction_id == 0:
      break

  translation = tokenizer.decode(dec_input_ids[0, 1:])
  return translation

In [136]:
def beam_search_translate(input_sentence, beam_width=5, max_len=32, length_penalty=0.6):
    """
    Translate using beam search decoding.

    Args:
        input_sentence: English sentence to translate
        beam_width: Number of hypotheses to keep (typically 3-10)
        max_len: Maximum translation length
        length_penalty: Alpha parameter (0 = no penalty, 0.6-0.7 is common)

    Returns:
        Best translation string and its score
    """
    transformer.eval()

    # Encode the input sentence once (shared for all beams)
    enc_input = tokenizer(input_sentence, return_tensors='pt').to(device)
    enc_output = encoder(enc_input['input_ids'], enc_input['attention_mask'])

    # Initialize beams: (sequence, log_probability)
    start_token = torch.tensor([[tokenizer.cls_token_id]], device=device)
    beams = [(start_token, 0.0)]  # (decoder_input_ids, cumulative_log_prob)
    completed_beams = []  # Store finished translations

    for step in range(max_len):
        all_candidates = []

        # Expand each beam
        for seq, score in beams:
            # Check if this beam is already complete
            if seq[0, -1].item() == tokenizer.eos_token_id:  # 0 is </s>
                completed_beams.append((seq, score))
                continue

            # Create attention mask for this beam
            dec_attn_mask = torch.ones_like(seq)

            # Get model predictions for the next token
            with torch.no_grad():
                dec_output = decoder(
                    enc_output,
                    seq,
                    enc_input['attention_mask'],
                    dec_attn_mask
                )

            # Get log probabilities for the last token
            last_token_logits = dec_output[:, -1, :]  # (1, vocab_size)
            log_probs = F.log_softmax(last_token_logits, dim=-1)  # Convert to log space

            # Get top k candidates (beam_width * 2 to maintain diversity)
            top_k_log_probs, top_k_indices = torch.topk(log_probs, beam_width * 2, dim=-1)

            # Create new candidates
            for i in range(beam_width * 2):
                new_token = top_k_indices[0, i].unsqueeze(0).unsqueeze(0)  # (1, 1)
                new_seq = torch.cat([seq, new_token], dim=-1)
                new_score = score + top_k_log_probs[0, i].item()  # Add log prob
                all_candidates.append((new_seq, new_score))

        # If all beams are complete, break early
        if not all_candidates and completed_beams:
            break

        # Select top k beams based on score
        all_candidates.sort(key=lambda x: x[1], reverse=True)  # Higher log prob is better
        beams = all_candidates[:beam_width]

    # Add any completed beams from the loop
    beams.extend(completed_beams)

    # Apply length penalty to avoid favoring shorter translations
    def apply_length_penalty(seq, score):
        """Normalize score by sequence length (excluding start token)"""
        length = seq.shape[1] - 1  # Exclude start token
        if length == 0:
            return score
        # Standard length penalty from Transformer paper
        return score / (length ** length_penalty)

    # Choose the best beam using length penalty
    best_seq, best_score = max(beams, key=lambda x: apply_length_penalty(x[0], x[1]))

    # Decode and return (skip the start token)
    translation = tokenizer.decode(best_seq[0, 1:])
    return translation, best_score

In [132]:
def compare_translations(sentences, beam_widths=[1, 3, 5]):
    """Compare greedy vs different beam widths"""

    results = []
    for sentence in sentences:
        print(f"\n{'='*60}")
        print(f"English: {sentence}")
        print(f"{'='*60}")

        # Greedy (beam_width=1)
        greedy_result = greedy_translate(sentence)
        print(f"Greedy (beam=1): {greedy_result}")

        # Different beam widths
        for bw in beam_widths:
            if bw == 1:
                continue
            beam_result, score = beam_search_translate(sentence, beam_width=bw)
            print(f"Beam (b={bw}):    {beam_result} (score: {score:.2f})")

        # Also show the expected translation if available
        # Find matching Russian sentence in dataset
        matching = df[df['en'] == sentence]['ru'].values
        if len(matching) > 0:
            print(f"Expected:        {matching[0]}")

    return results

In [150]:
def beam_search_verbose(input_sentence, beam_width=3, max_len=20):
    transformer.eval()

    enc_input = tokenizer(input_sentence, return_tensors='pt').to(device)
    enc_output = encoder(enc_input['input_ids'], enc_input['attention_mask'])

    start_token = torch.tensor([[tokenizer.cls_token_id]], device=device)
    beams = [(start_token, 0.0)]
    completed_beams = []  # ← ADD THIS!

    print(f"\n{'='*60}")
    print(f"Translating: '{input_sentence}'")
    print(f"Beam width: {beam_width}")
    print(f"{'='*60}\n")

    for step in range(max_len):
        print(f"Step {step + 1}:")
        all_candidates = []

        for beam_idx, (seq, score) in enumerate(beams):
            # If already finished, move to completed_beams
            if seq[0, -1].item() == tokenizer.eos_token_id:
                completed_beams.append((seq, score))
                continue

            dec_attn_mask = torch.ones_like(seq)

            with torch.no_grad():
                dec_output = decoder(enc_output, seq,
                                   enc_input['attention_mask'], dec_attn_mask)

            last_token_logits = dec_output[:, -1, :]
            log_probs = F.log_softmax(last_token_logits, dim=-1)
            top_k_log_probs, top_k_indices = torch.topk(log_probs, beam_width, dim=-1)

            for i in range(beam_width):
                new_token = top_k_indices[0, i].unsqueeze(0).unsqueeze(0)
                new_seq = torch.cat([seq, new_token], dim=-1)
                new_score = score + top_k_log_probs[0, i].item()
                all_candidates.append((new_seq, new_score, beam_idx))

        if not all_candidates:
            break

        # Sort and keep top beams
        all_candidates.sort(key=lambda x: x[1], reverse=True)
        beams = [(seq, score) for seq, score, _ in all_candidates[:beam_width]]

        # Print current beams
        for b_idx, (seq, score) in enumerate(beams):
            partial = tokenizer.decode(seq[0, 1:])
            if len(partial) > 40:
                partial = partial[:40] + "..."
            print(f"  Beam {b_idx}: '{partial}' (score: {score:.2f})")
        print()

    # Combine active and completed beams for final selection
    all_final_beams = beams + completed_beams  # ← ADD THIS!

    def length_penalty(seq, score, alpha=0.6):
        length = seq.shape[1] - 1
        return score / (length ** alpha) if length > 0 else score

    best_seq, best_score = max(all_final_beams, key=lambda x: length_penalty(x[0], x[1]))
    translation = tokenizer.decode(best_seq[0, 1:])

    print(f"{'='*60}")
    print(f"Best translation: {translation}")
    print(f"Score: {best_score:.2f}")
    print(f"{'='*60}")

    return translation

In [146]:
# Test sentences
test_sentences = [
    "How are you?",
    "I love you",
    "Good morning",
    "Thank you",
    "Where is the hotel?",
    "I would like to order coffee"
]

# Compare greedy vs beam search
print("COMPARING GREEDY VS BEAM SEARCH")
print("=" * 60)

for sentence in test_sentences:
    print(f"\nEnglish: {sentence}")
    print("-" * 40)

    # Greedy
    greedy_result = greedy_translate(sentence)
    print(f"Greedy:     {greedy_result}")

    # Beam search with width 3
    beam_result, score = beam_search_translate(sentence, beam_width=3)
    print(f"Beam (b=3): {beam_result} (score: {score:.2f})")

    # Beam search with width 5 (slower but potentially better)
    beam_result_5, score_5 = beam_search_translate(sentence, beam_width=5)
    print(f"Beam (b=5): {beam_result_5} (score: {score_5:.2f})")

# See the search process for one sentence
beam_search_verbose("I love you", beam_width=3)

COMPARING GREEDY VS BEAM SEARCH

English: How are you?
----------------------------------------
Greedy:     Как ты?</s>
Beam (b=3): Как ты?</s> (score: -0.63)
Beam (b=5): Как ты?</s> (score: -0.63)

English: I love you
----------------------------------------
Greedy:     Я люблю тебя любил.</s>
Beam (b=3): Я тебя люблю.</s> (score: -2.48)
Beam (b=5): Я тебя люблю.</s> (score: -2.48)

English: Good morning
----------------------------------------
Greedy:     Добродь жизнь!</s>
Beam (b=3): Спокойся жизнью.</s> (score: -7.29)
Beam (b=5): Хорошая правила.</s> (score: -4.45)

English: Thank you
----------------------------------------
Greedy:     Спасибо тебя.</s>
Beam (b=3): Спасибо тебя.</s> (score: -1.64)
Beam (b=5): Спасибо тебя.</s> (score: -1.64)

English: Where is the hotel?
----------------------------------------
Greedy:     Где выход?</s>
Beam (b=3): Где пошли?</s> (score: -2.97)
Beam (b=5): Где пошли?</s> (score: -2.97)

English: I would like to order coffee
---------------------

'Я люблю тебя любил. Пировал."а."аю тебя."а.</s>'

In [147]:
# res = greedy_translate("How are you?")
# res
sentence = "How are you?"
greedy_result = greedy_translate(sentence)
greedy_result
print(f"Greedy:     {greedy_result}")

Greedy:     Как ты?</s>


In [151]:
beam_search_verbose(sentence)


Translating: 'How are you?'
Beam width: 3

Step 1:
  Beam 0: 'Как' (score: -0.10)
  Beam 1: 'Сколько' (score: -3.01)
  Beam 2: 'Насколько' (score: -3.95)

Step 2:
  Beam 0: 'Как ты' (score: -0.43)
  Beam 1: 'Как вы' (score: -2.89)
  Beam 2: 'Как дела' (score: -3.16)

Step 3:
  Beam 0: 'Как ты?' (score: -0.63)
  Beam 1: 'Как дела?' (score: -3.32)
  Beam 2: 'Как вы знаете' (score: -3.65)

Step 4:
  Beam 0: 'Как ты?</s>' (score: -0.63)
  Beam 1: 'Как дела?</s>' (score: -3.32)
  Beam 2: 'Как вы знаете?' (score: -3.65)

Step 5:
  Beam 0: 'Как вы знаете?</s>' (score: -3.65)
  Beam 1: 'Как вы знаете? Как' (score: -11.63)
  Beam 2: 'Как вы знаете? Что' (score: -13.47)

Step 6:
  Beam 0: 'Как вы знаете? Как?' (score: -11.75)
  Beam 1: 'Как вы знаете? Что?' (score: -13.49)
  Beam 2: 'Как вы знаете? Как дела' (score: -14.29)

Step 7:
  Beam 0: 'Как вы знаете? Как?</s>' (score: -11.75)
  Beam 1: 'Как вы знаете? Что?</s>' (score: -13.49)
  Beam 2: 'Как вы знаете? Как дела?' (score: -14.30)

Step 8

'Как ты?</s>'

In [153]:
test_sentences = [
    "How are you?",
    "I love you",
    "Good morning",
    "Thank you",
    "Where is the hotel?",
    "I would like to order coffee"
]

compare_translations(test_sentences)


English: How are you?
Greedy (beam=1): Как ты?</s>
Beam (b=3):    Как ты?</s> (score: -0.63)
Beam (b=5):    Как ты?</s> (score: -0.63)
Expected:        Как поживаете?

English: I love you
Greedy (beam=1): Я люблю тебя любил.</s>
Beam (b=3):    Я тебя люблю.</s> (score: -2.48)
Beam (b=5):    Я тебя люблю.</s> (score: -2.48)

English: Good morning
Greedy (beam=1): Добродь жизнь!</s>
Beam (b=3):    Спокойся жизнью.</s> (score: -7.29)
Beam (b=5):    Хорошая правила.</s> (score: -4.45)

English: Thank you
Greedy (beam=1): Спасибо тебя.</s>
Beam (b=3):    Спасибо тебя.</s> (score: -1.64)
Beam (b=5):    Спасибо тебя.</s> (score: -1.64)

English: Where is the hotel?
Greedy (beam=1): Где выход?</s>
Beam (b=3):    Где пошли?</s> (score: -2.97)
Beam (b=5):    Где пошли?</s> (score: -2.97)

English: I would like to order coffee
Greedy (beam=1): Мне нравится очень кофе.</s>
Beam (b=3):    Мне нравится очень кофе.</s> (score: -3.22)
Beam (b=5):    Мне нравится очень кофе.</s> (score: -3.22)


[]